# 12_00 — Open candidate pool foundation: TRAIN + COCONUT

**Infrastructure only.** Nothing is downloaded, and nothing is trained or ranked here. You
obtain COCONUT **manually**, save it locally, and point `configs/v6/coconut_source.json` at it.
The schema (id, SMILES, name, formula columns) is configured there, never guessed.

Pipeline:

1. Read the local source in chunks. `casmi.candidates.sources` accepts CSV, TSV, Parquet,
   SMILES and SDF files.
2. Standardize each chunk with `casmi.candidates.standardize`. It uses the **training** structure
   table function: the same canonical SMILES, the same tautomer-canonical connectivity key and
   the same tautomer caps. Invalid structures go to an explicit **reject table** and are never
   dropped silently.
3. Deduplicate by connectivity with provenance kept (`candidate_sources`, `source_ids`), then
   merge with the TRAIN candidate library (`casmi.candidates.merge`) into one row per
   connectivity.
4. Build the mass index. It uses the inference retrieval arithmetic, persists to `.npy`, and is
   checked against brute force at runtime. A formula index is built as metadata only.
5. Build the source-prior **schema**. Only membership flags are filled; no invented values and
   no ranking weights.
6. Lay the Class-2 foundation. A manifest is written in which the truth stays in the pool, its
   references are hidden and its TRAIN provenance is masked. No fingerprint model is built.
   Class 3 is documented only.

Outputs go to `outputs/v6/candidates/`.

In [1]:
import sys, json, hashlib
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd

from casmi.paths import get_project_paths
from casmi.qcr.context import v5_paths
from casmi.qcr.fingerprint import code_version
from casmi.candidates import merge as cm
from casmi.candidates.mass_index import FormulaIndex, OpenMassIndex, open_index_parity
from casmi.candidates.provenance import EVIDENCE_SCHEMA, FORBIDDEN_SHORTCUT_FEATURES, source_prior_frame
from casmi.candidates.sources import SourceNotSupplied, config_record, file_fingerprint, iter_source_records, load_source_config
from casmi.candidates.standardize import standardize_in_chunks, training_tautomer_caps
from casmi.validation import class2_split as c2

paths = get_project_paths()
D = v5_paths(paths)
OUT = paths.outputs / 'v6' / 'candidates'
OUT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = REPO_ROOT / 'configs' / 'v6' / 'coconut_source.json'
N_JOBS = 8                       # RDKit tautomer canonicalization is the slow step; chunks are resumable
cfg = load_source_config(CONFIG_PATH)
if not cfg.path or not Path(cfg.path).exists():
    msg = (f'COCONUT is not supplied: set "path" in {CONFIG_PATH} to a LOCAL file you obtained manually '
           f'(and verify id_col / smiles_col / name_col / formula_col against its header). This notebook never downloads.')
    print(msg)
    raise SourceNotSupplied(msg)
SRC_SHA = file_fingerprint(cfg.path)
print('source:', cfg.source, '|', cfg.path, '| format:', cfg.format, '| sha256:', SRC_SHA[:16], '| tautomer caps (training):', training_tautomer_caps())

COCONUT is not supplied: set "path" in c:\Users\myben\OneDrive\Documents\Enveda\configs\v6\coconut_source.json to a LOCAL file you obtained manually (and verify id_col / smiles_col / name_col / formula_col against its header). This notebook never downloads.


SourceNotSupplied: COCONUT is not supplied: set "path" in c:\Users\myben\OneDrive\Documents\Enveda\configs\v6\coconut_source.json to a LOCAL file you obtained manually (and verify id_col / smiles_col / name_col / formula_col against its header). This notebook never downloads.

## 1. Read + standardize (chunked, resumable) → `coconut_standardized.parquet`, `coconut_rejected.parquet`

In [ ]:
CHUNK_DIR = OUT / '_chunks' / f'{cfg.source}_{SRC_SHA[:12]}'          # a different input file never reuses chunks
n_input = {'n': 0}


def counted(chunks):
    for c in chunks:
        n_input['n'] += len(c)
        yield c


STD, REJ = standardize_in_chunks(counted(iter_source_records(cfg)), CHUNK_DIR, n_jobs=N_JOBS)
assert len(STD) + len(REJ) == n_input['n'], 'every input record is standardized or rejected'
STD.to_parquet(OUT / 'coconut_standardized.parquet', index=False)
REJ.to_parquet(OUT / 'coconut_rejected.parquet', index=False)
print(f"{n_input['n']:,} records -> {len(STD):,} standardized ({STD['connectivity_key'].nunique():,} connectivities), {len(REJ):,} rejected")
display(REJ['failure_reason'].str.split(':').str[0].value_counts().rename('n_rejected'))
print('flags: multi-fragment', int((STD['n_fragments'] > 1).sum()), '| charged', int(STD['is_charged'].sum()),
      '| tautomer cap hit', int(STD['tautomer_hit_cap'].fillna(False).astype(bool).sum()))

## 2. Unified TRAIN + COCONUT universe (one row per connectivity, provenance kept) → `unified_structures.parquet`

In [ ]:
mmv = pd.read_parquet(paths.interim / 'candidate_generation' / 'molecule_mass_variants.parquet')
UNI, VARIANTS = cm.unify(mmv, STD)
UNI.to_parquet(OUT / 'unified_structures.parquet', index=False)
VARIANTS.to_parquet(OUT / 'unified_mass_variants.parquet', index=False)
SUMMARY = cm.source_summary(UNI, STD, REJ, {cfg.canonical_source(): n_input['n']})
SUMMARY.to_csv(OUT / 'candidate_source_summary.csv', index=False)
display(SUMMARY)
shared = UNI[UNI['in_train'] & UNI['open_representative_smiles'].notna()]
print(f"TRAIN connectivities: {int(UNI['in_train'].sum()):,} | also in {cfg.source}: {len(shared):,} | "
      f"{cfg.source}-only (new candidates): {int((~UNI['in_train']).sum()):,}")
print('TRAIN vs external mass |diff| (Da) on shared connectivities:', shared['train_vs_open_mass_diff_da'].describe().round(6).to_dict())
assert UNI['connectivity_key'].is_unique and UNI['source_ids'].map(len).ge(1).all()

## 3. Mass index (inference arithmetic) + brute-force parity, and TRAIN retrieval unchanged → `mass_index.npy`, `mass_index_keys.npy`

In [ ]:
IDX = OpenMassIndex.from_variants(VARIANTS)
idx_meta = IDX.save(OUT, extra={'unified_sha256': hashlib.sha256((OUT / 'unified_structures.parquet').read_bytes()).hexdigest()})
rng = np.random.default_rng(42)
q_masses = np.concatenate([rng.choice(VARIANTS['exact_mass'].to_numpy(float), size=200, replace=False), [VARIANTS['exact_mass'].max() + 1e4, 1e-3]])
sample = rng.choice(len(VARIANTS), size=min(200_000, len(VARIANTS)), replace=False)      # brute force on a sub-library keeps this fast
sub = VARIANTS.iloc[np.sort(sample)]
PAR = open_index_parity(OpenMassIndex.from_variants(sub), sub['connectivity_key'].to_numpy(), sub['exact_mass'].to_numpy(float), q_masses)
print('open index vs brute force:', int(PAR['keys_equal'].sum()), '/', len(PAR), 'equal; max ppm diff', PAR['ppm_max_abs_diff'].max())
assert PAR['keys_equal'].all() and (PAR['ppm_max_abs_diff'] == 0).all()

# the closed-world TRAIN retrieval is untouched: TRAIN-only variants through OpenMassIndex == the exported bundle's MassSearch
if (D['bundle'] / 'structures.parquet').exists():
    from casmi_infer.mass_search import MassSearch
    bst = pd.read_parquet(D['bundle'] / 'structures.parquet')
    bconn = pd.read_parquet(D['bundle'] / 'connectivities.parquet')
    ms = MassSearch(bst['neutral_monoisotopic_mass'].to_numpy(float), bst['conn_idx'].to_numpy(), bconn['connectivity_key'].to_numpy())
    tr_idx = OpenMassIndex.from_variants(VARIANTS[VARIANTS['variant_source'] == 'TRAIN'])
    bad = 0
    for q in q_masses[:200]:
        ci, ap = ms._dedupe(np.arange(*ms.window(q, 50.0)), q)
        k, a = tr_idx.search(q, 50.0)
        bad += int(not (np.array_equal(bconn['connectivity_key'].to_numpy()[ci].astype(str), k) and np.array_equal(ap, a)))
    print('TRAIN-only open index vs bundle retrieval mismatches (200 masses, 50 ppm):', bad)
    assert bad == 0

## 4. Formula index (METADATA ONLY) + source-prior schema (no values invented, no weights)

In [ ]:
FIDX = FormulaIndex.from_unified(UNI)
assert FormulaIndex.HARD_FILTER is False
PRIORS = source_prior_frame(UNI)
PRIORS.to_parquet(OUT / 'source_priors_schema.parquet', index=False)
print(f'formula index: {len(FIDX):,} formulas (metadata only, never a retrieval filter)')
display(PRIORS.drop(columns='connectivity_key').describe(include='all').T)
print('future evidence schema:', list(EVIDENCE_SCHEMA))
print('forbidden shortcut features:', FORBIDDEN_SHORTCUT_FEATURES)

## 5. Class-2 foundation (split / manifest only; no fingerprint model)

A development query is Class-2-eligible when its truth is in the unified universe **and** in a
non-TRAIN source. Only then can it be shown as an ordinary external structure. Hiding is
fold-scoped. The check asserts four things: the truth is in the pool, no accessible reference
belongs to a hidden truth, no TRAIN or reference-availability trace remains, and no shortcut
feature is present. **Class 3** is out of scope: the truth would be removed from the pool.

In [ ]:
from casmi.io.artifacts import load_artifact
from casmi.ranking.manifests import load_manifest
ref_conn = load_artifact('train_spectrum_metadata')['connectivity_key'].astype(str).to_numpy()
C2 = {}
for name in ('TL_EVAL', 'MOL_DEV'):
    try:
        man = load_manifest(name, D['manifests'])[0]
    except FileNotFoundError:
        print(f'{name}: manifest missing -- skipped'); continue
    elig = c2.class2_eligible(man, UNI)
    mf = c2.class2_manifest(elig, ref_conn, hidden_scope='fold')
    mf.to_parquet(OUT / f'class2_manifest_{name}.parquet', index=False)
    hidden = set(mf['truth_connectivity_key'])
    view = c2.class2_candidate_view(UNI, hidden)
    allowed = c2.class2_reference_mask(ref_conn, hidden)
    c2.assert_class2_split(mf, view, ref_conn, allowed, feature_names=[])
    C2[name] = {'n_queries': int(len(man)), 'n_class2_eligible': int(elig['class2_eligible'].sum()), 'n_hidden_connectivities': len(hidden),
                'n_hidden_reference_spectra': int((~allowed).sum())}
    print(name, C2[name])
print(c2.CLASS3_DEFINITION)

## 6. Build manifest → `build_manifest.json`

In [ ]:
BUILD = {'created_at': datetime.now(timezone.utc).isoformat(), 'code_version': code_version(), 'source_config': config_record(cfg),
         'source_file_sha256': SRC_SHA, 'tautomer_caps': training_tautomer_caps(), 'n_input_records': n_input['n'],
         'n_standardized': int(len(STD)), 'n_rejected': int(len(REJ)), 'n_unified_connectivities': int(len(UNI)),
         'n_train_connectivities': int(UNI['in_train'].sum()), 'n_new_connectivities': int((~UNI['in_train']).sum()),
         'mass_index': idx_meta, 'formula_index': {'n_formulas': len(FIDX), 'hard_filter': False}, 'class2': C2,
         'outputs': sorted(p.name for p in OUT.glob('*') if p.is_file()),
         'policy': {'download': 'never (user-supplied local file)', 'formula': 'metadata only', 'source_priors': 'schema only, no weights',
                    'has_reference_spectrum': 'metadata only; forbidden as a raw ranker feature'}}
(OUT / 'build_manifest.json').write_text(json.dumps(BUILD, indent=2, default=str), encoding='utf-8')
print(json.dumps({k: BUILD[k] for k in ('n_input_records', 'n_standardized', 'n_rejected', 'n_unified_connectivities', 'n_train_connectivities',
                                        'n_new_connectivities', 'class2')}, indent=1))